# FoodTrack: esquema relacional en SQL Server
**Homework M2 · L1 (Henry)**

Este notebook **ejecuta paso a paso los scripts de la carpeta `scripts/`** y te explica qué hace cada uno.
La idea es que no solo se cree la base, sino que entiendas cada parte.

**Cómo se usa:** ejecutá las celdas en orden (`Shift + Enter`). Cada paso tiene:
- 🟦 una explicación de **qué hacemos y por qué**,
- ▶️ una celda que **muestra el script y lo ejecuta**,
- ✅ una celda de **verificación** para comprobar que quedó bien.

| Paso | Script | Qué hace |
|---|---|---|
| 0 | `00_create_database.sql` | Crea la base `foodtrack` |
| 1 | `01_create_tables.sql` | Crea las 5 tablas con claves primarias y restricciones |
| 2 | `02_add_foreign_keys.sql` | Une las tablas con claves foráneas |
| 3 | `03_alter_orders_add_comentarios.sql` | Cambia el esquema (agrega una columna) |

**Estructura del repositorio**
```
foodtrack-db/
├── README.md
├── foodtrack_homework.ipynb
├── data/       ← los 5 CSV de la consigna
└── scripts/    ← los scripts SQL
```

## Conexión a SQL Server

**Qué hacemos:** nos conectamos al contenedor de Docker donde corre SQL Server.

- Host `localhost`, puerto **1434** (el contenedor mapea el 1434 de tu Mac al 1433 de adentro).
- Usuario y contraseña: los lee del archivo **`.env`** que está una carpeta más arriba (`Modulo 2/.env`), el mismo que usa Docker Compose. Así las credenciales viven en **un solo lugar**.
- Ese `.env` queda **fuera** de esta carpeta (la que se sube a GitHub) y además está en `.gitignore`: la contraseña nunca viaja al repositorio.
- Si no encuentra el `.env`, el notebook te pide la contraseña por pantalla.

La celda define estas funciones, que usamos en todo el notebook:

| Función | Para qué sirve |
|---|---|
| `correr_script(nombre)` | **Muestra** un script de `scripts/` y lo **ejecuta** |
| `ver(sql)` | Corre un `SELECT` y devuelve una tabla para mirar |
| `ejecutar(sql)` | Corre una sentencia suelta que modifica la base |
| `probar_que_falla(sql, restriccion)` | Corre un `INSERT` que **debería** fallar y revisa que lo haya frenado esa restricción |

In [1]:
import getpass
from pathlib import Path

import pandas as pd
import pymssql

HOST = "localhost"
PORT = 1434
CARPETA_SCRIPTS = Path("scripts")
CARPETA_DATA = Path("data")


def leer_env(ruta):
    """Lee un archivo .env (líneas CLAVE=valor) y devuelve un diccionario."""
    datos = {}
    if ruta.exists():
        for linea in ruta.read_text(encoding="utf-8").splitlines():
            linea = linea.strip()
            if linea and not linea.startswith("#") and "=" in linea:
                clave, valor = linea.split("=", 1)
                datos[clave.strip()] = valor.strip().strip('"').strip("'")
    return datos


env = leer_env(Path("..") / ".env")  # el .env está una carpeta más arriba
USUARIO = env.get("MSSQL_USER", "sa")
PASSWORD = env.get("MSSQL_SA_PASSWORD") or getpass.getpass("No encontré la contraseña en .env. Contraseña de sa: ")


def conectar(base="master"):
    """Abre una conexión. autocommit=True: cada sentencia se aplica al instante."""
    return pymssql.connect(server=HOST, port=PORT, user=USUARIO,
                           password=PASSWORD, database=base, autocommit=True)


def ejecutar(sql, base="foodtrack"):
    """Ejecuta SQL que modifica la base (CREATE, ALTER, INSERT...)."""
    conn = conectar(base)
    try:
        conn.cursor().execute(sql)
    finally:
        conn.close()


def ver(sql, base="foodtrack"):
    """Ejecuta un SELECT y devuelve el resultado como tabla de pandas."""
    conn = conectar(base)
    try:
        cur = conn.cursor()
        cur.execute(sql)
        columnas = [c[0] for c in cur.description]
        filas = cur.fetchall()
    finally:
        conn.close()
    return pd.DataFrame(filas, columns=columnas)


def correr_script(nombre, base="foodtrack"):
    """Muestra el contenido de scripts/<nombre> y lo ejecuta."""
    sql = (CARPETA_SCRIPTS / nombre).read_text(encoding="utf-8")
    print(f"📄 scripts/{nombre}\n" + "-" * 60)
    print(sql)
    print("-" * 60)
    ejecutar(sql, base=base)
    print("✅ Script ejecutado sin errores")


def probar_que_falla(sql, restriccion):
    """Ejecuta un INSERT que DEBERÍA fallar y comprueba que lo frenó `restriccion`.

    Hay tres resultados posibles:
    - el INSERT entra sin error: la restricción no está funcionando.
    - falla y el error nombra `restriccion`: la prueba pasa.
    - falla por otro motivo (otra restricción, una columna mal escrita...):
      la prueba no está probando lo que creemos.
    """
    try:
        ejecutar(sql)
    except Exception as e:
        mensaje = str(e)
        if restriccion in mensaje:
            print(f"❌ Falló por {restriccion}, como esperábamos:")
        else:
            print(f"⚠️ Falló, pero NO por {restriccion}. Revisá el error:")
        print(mensaje[:250])
        return
    print(f"⚠️ Se insertó: {restriccion} no está funcionando")

**Probamos la conexión.** Si todo está bien, muestra la versión de SQL Server. Si da error, revisá que el contenedor esté corriendo (`docker ps`) y la contraseña del `.env`.

In [2]:
ver("SELECT @@VERSION AS version", base="master")

,version
0,Microsoft SQL Server 2025 (RTM-CU9) (KB5122048...


## Antes de diseñar: mirar los datos

**Qué hacemos:** la consigna dice *"analizar los CSV y diseñar el modelo a partir de los datos"*. Todo diseño empieza mirando qué hay.

Fijate en las columnas que terminan en `_id`. Cuando una columna con el **mismo nombre** aparece en dos archivos, ahí hay una **relación**:

- `foodtruck_id` está en `locations`, `products` y `orders` → un foodtruck tiene muchas ubicaciones, productos y pedidos.
- `order_id` está en `order_items` → un pedido tiene muchos ítems.
- `product_id` está en `order_items` → un producto aparece en muchos ítems.

In [3]:
tablas_csv = ["foodtrucks", "locations", "products", "orders", "order_items"]
datos = {t: pd.read_csv(CARPETA_DATA / f"{t}.csv") for t in tablas_csv}

for nombre, df in datos.items():
    print(f"=== {nombre}.csv  ({len(df)} filas)")
    display(df)

=== foodtrucks.csv  (2 filas)


,foodtruck_id,name,cuisine_type,city
0,1,Taco Loco,Mexicana,Ciudad de México
1,2,Burger Bros,Americana,Buenos Aires


=== locations.csv  (2 filas)


,location_id,foodtruck_id,location_date,zone
0,1,1,2023-09-01,Centro
1,2,2,2023-09-01,Parque


=== products.csv  (4 filas)


,product_id,foodtruck_id,name,price,stock
0,101,1,Taco al pastor,50,100
1,102,1,Quesadilla,40,80
2,103,2,Cheeseburger,70,120
3,104,2,Papas fritas,30,150


=== orders.csv  (2 filas)


,order_id,foodtruck_id,order_date,status,total
0,1001,1,2023-09-01,entregado,90
1,1002,2,2023-09-01,pendiente,100


=== order_items.csv  (3 filas)


,order_item_id,order_id,product_id,quantity
0,1,1001,101,1
1,2,1002,103,1
2,3,1002,104,1


Dos cosas que nos dicen estos datos y que van a influir en el diseño:

1. **Los IDs no empiezan en 1** (los pedidos arrancan en `1001`, los productos en `101`). Por eso los definimos como `INT` normales y no como `IDENTITY` (autonumérico): así podemos cargar los CSV tal cual vienen.
2. **`orders.status` trae texto completo** (`entregado`, `pendiente`). Nosotros lo vamos a **codificar con una letra** (`P`, `R`, `E`, `C`). Lo vemos en el paso 1.

## Paso 0 · Crear la base de datos
**Script:** `00_create_database.sql`

**Qué hacemos:** una *base de datos* es el contenedor donde viven las tablas. SQL Server trae bases internas (`master`, `tempdb`...) y **no conviene crear tablas ahí**, así que creamos una propia.

**Qué hace cada parte:**
- `IF DB_ID('foodtrack') IS NULL` → pregunta *"¿ya existe una base llamada foodtrack?"*. Así el script se puede correr más de una vez sin dar error.
- `CREATE DATABASE foodtrack;` → la crea.

Este es el único script que se conecta a `master`, porque `foodtrack` todavía no existe.

In [4]:
correr_script("00_create_database.sql", base="master")

📄 scripts/00_create_database.sql
------------------------------------------------------------
-- FoodTrack: crea la base de datos del proyecto (solo si todavía no existe).
-- Se ejecuta conectado a master, porque foodtrack aún no existe.
IF DB_ID('foodtrack') IS NULL
    CREATE DATABASE foodtrack;

------------------------------------------------------------


✅ Script ejecutado sin errores


**✅ Verificamos** que la base exista:

In [5]:
ver("SELECT name FROM sys.databases WHERE name = 'foodtrack'", base="master")

,name
0,foodtrack


## Paso 1 · Crear las tablas
**Script:** `01_create_tables.sql`

**Qué hacemos:** una tabla es una grilla de **columnas** (cada una con un tipo de dato) y **filas**. Al crearla definimos también reglas (*restricciones*) para que no entren datos inválidos.

### Conceptos del script

| Concepto | Qué hace | Dónde lo usamos |
|---|---|---|
| **PRIMARY KEY** | Identifica cada fila. No se repite ni es nula | `foodtruck_id`, `order_id`... en cada tabla |
| **NOT NULL** | La columna no puede quedar vacía | casi todas las columnas |
| **UNIQUE** | No se repite el valor (o la combinación) | `foodtrucks.name`; `(foodtruck_id, location_date)` en `locations`; `(foodtruck_id, name)` en `products`; `(order_id, product_id)` en `order_items` |
| **CHECK** | Una condición que el dato debe cumplir | `price >= 0`, `stock >= 0`, `total >= 0`, `quantity > 0`, `status` |

### Tipos de dato

| Tipo | Para qué |
|---|---|
| `INT` | Enteros: IDs y cantidades |
| `NVARCHAR(n)` | Texto de hasta *n* caracteres. La **N** admite acentos ("México") |
| `DECIMAL(10,2)` | Dinero: 10 dígitos, 2 decimales. No usamos `FLOAT` porque redondea mal |
| `DATE` | Fechas |
| `CHAR(1)` | Un carácter: el código de estado del pedido |

### Decisiones de diseño
- **`status` codificado:** `P` = pendiente, `R` = en preparación, `E` = entregado, `C` = cancelado. Ocupa 1 carácter y el `CHECK` evita códigos inventados.
- **UNIQUE `(foodtruck_id, location_date)`:** un foodtruck no puede estar en dos zonas el mismo día.
- **UNIQUE `(order_id, product_id)`:** un producto aparece una sola vez por pedido. Si alguien pide 3 tacos, es una fila con `quantity = 3`, no tres filas.
- **Se puede correr más de una vez:** cada `CREATE TABLE` va dentro de `IF OBJECT_ID('tabla', 'U') IS NULL` (*"si la tabla no existe"*). Sin eso, la segunda vez da error porque la tabla ya existe.
- **Restricciones con nombre** (`pk_`, `uq_`, `ck_`): cuando algo falla, el error dice cuál se rompió.
- **Sin claves foráneas todavía:** primero las tablas, después las relaciones. Es un avance estructural por separado (un commit por cada uno).

In [6]:
correr_script("01_create_tables.sql")

📄 scripts/01_create_tables.sql
------------------------------------------------------------
-- FoodTrack: creación de tablas con claves primarias y restricciones.
-- Las claves foráneas se agregan aparte en 02_add_foreign_keys.sql.
-- Cada tabla se crea solo si todavía no existe, así el script se puede correr más de una vez.
USE foodtrack;

IF OBJECT_ID('foodtrucks', 'U') IS NULL
    CREATE TABLE foodtrucks (
        foodtruck_id INT           NOT NULL,
        name         NVARCHAR(100) NOT NULL,
        cuisine_type NVARCHAR(50)  NOT NULL,
        city         NVARCHAR(100) NOT NULL,
        CONSTRAINT pk_foodtrucks PRIMARY KEY (foodtruck_id),
        CONSTRAINT uq_foodtrucks_name UNIQUE (name)
    );

IF OBJECT_ID('locations', 'U') IS NULL
    CREATE TABLE locations (
        location_id   INT           NOT NULL,
        foodtruck_id  INT           NOT NULL,
        location_date DATE          NOT NULL,
        zone          NVARCHAR(100) NOT NULL,
        CONSTRAINT pk_locations PR

✅ Script ejecutado sin errores


**✅ Verificación 1:** ¿están las 5 tablas?

In [7]:
ver("""
SELECT TABLE_NAME AS tabla
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_TYPE = 'BASE TABLE'
ORDER BY TABLE_NAME
""")

,tabla
0,foodtrucks
1,locations
2,order_items
3,orders
4,products


**✅ Verificación 2:** columnas y tipos de cada tabla. Compará con lo que viste en los CSV.

In [8]:
ver("""
SELECT TABLE_NAME AS tabla, COLUMN_NAME AS columna, DATA_TYPE AS tipo,
       CHARACTER_MAXIMUM_LENGTH AS largo, IS_NULLABLE AS acepta_nulos
FROM INFORMATION_SCHEMA.COLUMNS
ORDER BY TABLE_NAME, ORDINAL_POSITION
""")

,tabla,columna,tipo,largo,acepta_nulos
0,foodtrucks,foodtruck_id,int,NaN,NO
1,foodtrucks,name,nvarchar,100.0,NO
2,foodtrucks,cuisine_type,nvarchar,50.0,NO
3,foodtrucks,city,nvarchar,100.0,NO
4,locations,location_id,int,NaN,NO
5,locations,foodtruck_id,int,NaN,NO
6,locations,location_date,date,NaN,NO
7,locations,zone,nvarchar,100.0,NO
8,order_items,order_item_id,int,NaN,NO
9,order_items,order_id,int,NaN,NO


**✅ Verificación 3:** las restricciones creadas (claves primarias, únicas y `CHECK`):

In [9]:
ver("""
SELECT TABLE_NAME AS tabla, CONSTRAINT_NAME AS restriccion, CONSTRAINT_TYPE AS tipo
FROM INFORMATION_SCHEMA.TABLE_CONSTRAINTS
ORDER BY TABLE_NAME, CONSTRAINT_TYPE
""")

,tabla,restriccion,tipo
0,foodtrucks,pk_foodtrucks,PRIMARY KEY
1,foodtrucks,uq_foodtrucks_name,UNIQUE
2,locations,pk_locations,PRIMARY KEY
3,locations,uq_locations_truck_date,UNIQUE
4,order_items,ck_order_items_quantity,CHECK
5,order_items,pk_order_items,PRIMARY KEY
6,order_items,uq_order_items_order_product,UNIQUE
7,orders,ck_orders_status,CHECK
8,orders,ck_orders_total,CHECK
9,orders,pk_orders,PRIMARY KEY


### 🧪 Probamos que un `CHECK` funciona
Una restricción que nunca viste fallar es una restricción que no sabés si anda. Intentamos insertar un producto con **precio negativo**. Tiene que dar error, y eso es lo correcto.

*(Si corrés el notebook sobre una base que ya tiene claves foráneas, este producto además apunta a un foodtruck que no existe. Por eso no alcanza con que falle: `probar_que_falla` revisa que lo haya frenado justo `ck_products_price`.)*

In [10]:
probar_que_falla(
    "INSERT INTO products (product_id, foodtruck_id, name, price, stock) "
    "VALUES (1, 1, 'Prueba', -5, 10)",
    restriccion="ck_products_price",
)

❌ Falló por ck_products_price, como esperábamos:
(547, b'The INSERT statement conflicted with the CHECK constraint "ck_products_price". The conflict occurred in database "foodtrack", table "dbo.products", column \'price\'.DB-Lib error message 20018, severity 16:\nGeneral SQL Server error: Check mes


## Paso 2 · Relaciones (claves foráneas)
**Script:** `02_add_foreign_keys.sql`

**Qué hacemos:** una **clave foránea** obliga a que el valor de una columna **exista** en otra tabla. Ejemplo: no se puede crear un pedido del `foodtruck_id = 99` si ese foodtruck no existe. Esto se llama **integridad referencial**.

Se agregan a tablas ya creadas con `ALTER TABLE ... ADD CONSTRAINT ... FOREIGN KEY ... REFERENCES ...`. Cada una va dentro de `IF OBJECT_ID('fk_...', 'F') IS NULL`, así el script se puede correr más de una vez.

**Las 5 relaciones:**

| Tabla hija | Columna | Apunta a |
|---|---|---|
| `locations` | `foodtruck_id` | `foodtrucks(foodtruck_id)` |
| `products` | `foodtruck_id` | `foodtrucks(foodtruck_id)` |
| `orders` | `foodtruck_id` | `foodtrucks(foodtruck_id)` |
| `order_items` | `order_id` | `orders(order_id)` |
| `order_items` | `product_id` | `products(product_id)` |

La tabla **hija** es la que tiene la clave foránea; la **padre** es a la que apunta.

In [11]:
correr_script("02_add_foreign_keys.sql")

📄 scripts/02_add_foreign_keys.sql
------------------------------------------------------------
-- FoodTrack: relaciones entre tablas (claves foráneas).
-- Se ejecuta después de 01_create_tables.sql.
-- Cada clave foránea se agrega solo si todavía no existe, así el script se puede correr más de una vez.
USE foodtrack;

IF OBJECT_ID('fk_locations_foodtruck', 'F') IS NULL
    ALTER TABLE locations
        ADD CONSTRAINT fk_locations_foodtruck
        FOREIGN KEY (foodtruck_id) REFERENCES foodtrucks (foodtruck_id);

IF OBJECT_ID('fk_products_foodtruck', 'F') IS NULL
    ALTER TABLE products
        ADD CONSTRAINT fk_products_foodtruck
        FOREIGN KEY (foodtruck_id) REFERENCES foodtrucks (foodtruck_id);

IF OBJECT_ID('fk_orders_foodtruck', 'F') IS NULL
    ALTER TABLE orders
        ADD CONSTRAINT fk_orders_foodtruck
        FOREIGN KEY (foodtruck_id) REFERENCES foodtrucks (foodtruck_id);

IF OBJECT_ID('fk_order_items_order', 'F') IS NULL
    ALTER TABLE order_items
        ADD CONSTRAI

✅ Script ejecutado sin errores


**✅ Verificación:** tienen que aparecer 5 relaciones.

In [12]:
ver("""
SELECT fk.name AS relacion,
       OBJECT_NAME(fk.parent_object_id)     AS tabla_hija,
       OBJECT_NAME(fk.referenced_object_id) AS tabla_padre
FROM sys.foreign_keys fk
ORDER BY tabla_hija
""")

,relacion,tabla_hija,tabla_padre
0,fk_locations_foodtruck,locations,foodtrucks
1,fk_order_items_order,order_items,orders
2,fk_order_items_product,order_items,products
3,fk_orders_foodtruck,orders,foodtrucks
4,fk_products_foodtruck,products,foodtrucks


### 🧪 Probamos que la relación funciona
Intentamos insertar un pedido de un foodtruck que **no existe** (`foodtruck_id = 99`). Tiene que dar error por la clave foránea `fk_orders_foodtruck`.

Las columnas van escritas a propósito: con `INSERT INTO orders VALUES (...)` a secas, después del paso 3 (cuando `orders` ya tiene 6 columnas) fallaría por la cantidad de valores y no por la clave foránea.

In [13]:
probar_que_falla(
    "INSERT INTO orders (order_id, foodtruck_id, order_date, status, total) "
    "VALUES (9999, 99, '2023-09-01', 'P', 10)",
    restriccion="fk_orders_foodtruck",
)

❌ Falló por fk_orders_foodtruck, como esperábamos:
(547, b'The INSERT statement conflicted with the FOREIGN KEY constraint "fk_orders_foodtruck". The conflict occurred in database "foodtrack", table "dbo.foodtrucks", column \'foodtruck_id\'.DB-Lib error message 20018, severity 16:\nGeneral SQL Server


## Paso 3 · Evolución del esquema
**Script:** `03_alter_orders_add_comentarios.sql`

**Qué hacemos:** los esquemas cambian con el tiempo. La consigna pide **una modificación estructural** versionada en su propio commit.

El cambio: agregar a `orders` una columna `comentarios` para notas del pedido (por ejemplo "sin cebolla").

**Qué hace cada parte:**
- `ALTER TABLE orders` → modifica una tabla que ya existe.
- `IF COL_LENGTH('orders', 'comentarios') IS NULL` → solo la agrega si todavía no existe (así el script se puede correr más de una vez).
- `ADD comentarios NVARCHAR(255) NULL` → agrega la columna. Es `NULL` (opcional) a propósito: los pedidos que ya existen no tienen comentarios, y si fuera obligatoria no se podría agregar.

In [14]:
correr_script("03_alter_orders_add_comentarios.sql")

📄 scripts/03_alter_orders_add_comentarios.sql
------------------------------------------------------------
-- FoodTrack: evolución del esquema.
-- Agrega la columna opcional "comentarios" a orders (acepta nulos).
-- Solo la agrega si todavía no existe, así el script se puede correr más de una vez.
USE foodtrack;

IF COL_LENGTH('orders', 'comentarios') IS NULL
    ALTER TABLE orders
        ADD comentarios NVARCHAR(255) NULL;

------------------------------------------------------------


✅ Script ejecutado sin errores


**✅ Verificamos** que la columna nueva exista en `orders`:

In [15]:
ver("""
SELECT COLUMN_NAME AS columna, DATA_TYPE AS tipo, IS_NULLABLE AS acepta_nulos
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_NAME = 'orders'
ORDER BY ORDINAL_POSITION
""")

,columna,tipo,acepta_nulos
0,order_id,int,NO
1,foodtruck_id,int,NO
2,order_date,date,NO
3,status,char,NO
4,total,decimal,NO
5,comentarios,nvarchar,YES


## 🧹 (Opcional) Empezar de cero

Si querés repetir todo desde el paso 1, esta celda borra las 5 tablas. Se borran en este orden (primero las hijas) porque una clave foránea impide borrar una tabla que otra necesita.

⚠️ Borra datos. Está comentada a propósito: sacá los `#` para usarla.

In [16]:
# ejecutar("""
# DROP TABLE IF EXISTS order_items;
# DROP TABLE IF EXISTS orders;
# DROP TABLE IF EXISTS products;
# DROP TABLE IF EXISTS locations;
# DROP TABLE IF EXISTS foodtrucks;
# """)

## Git y GitHub: un commit por avance

Cada script es un avance estructural, así que cada uno va en su commit. Mensajes propuestos:

| # | Qué se agrega | Mensaje de commit |
|---|---|---|
| 1 | `README.md`, `.gitignore`, `data/` | `Agrega README y CSV de datos` |
| 2 | `scripts/00_create_database.sql` | `Agrega script de creación de la base foodtrack` |
| 3 | `scripts/01_create_tables.sql` | `Crea tablas con claves primarias y restricciones` |
| 4 | `scripts/02_add_foreign_keys.sql` | `Agrega claves foráneas entre tablas` |
| 5 | `scripts/03_alter_orders_add_comentarios.sql` | `Agrega columna comentarios a orders` |
| 6 | `foodtrack_homework.ipynb` | `Agrega notebook que ejecuta los scripts paso a paso` |

El repositorio en GitHub debe ser **público** y llamarse **`foodtrack-db`**.

## Extra credits (optativo)
- Cargar los CSV en las tablas (`BULK INSERT` o Python con `pymssql`/`pyodbc`). Ojo: el `orders.csv` trae `entregado`/`pendiente` y la tabla espera `E`/`P`, así que hay que traducir al cargar.
- Registrar los errores de inserción en una tabla auxiliar (`failed_orders`).
- Diagrama ER: ya está en el `README.md`.